## Load dataset from GitHub (demonstration)

In a normal Python or paid Databricks environment, we could download the dataset directly from GitHub using Python's urllib.  
This demonstrates how we would load the CSV without uploading it manually.  

**Note:** In Databricks Community Edition, this approach does **not work** because we cannot write files directly to DBFS for Spark to access.  
In this environment, the CSV must be uploaded manually to `/FileStore/tables/`.



In [0]:
import urllib.request
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define schema
schema = StructType([
    StructField("Country", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Exercise Level", StringType(), True),
    StructField("Diet Type", StringType(), True),
    StructField("Sleep Hours", DoubleType(), True),
    StructField("Stress Level", StringType(), True),
    StructField("Mental Health Condition", StringType(), True),
    StructField("Work Hours per Week", IntegerType(), True),
    StructField("Screen Time per Day (Hours)", DoubleType(), True),
    StructField("Social Interaction Score", DoubleType(), True),
    StructField("Happiness Score", DoubleType(), True)
])

# URL of the CSV in GitHub
url = "https://raw.githubusercontent.com/jorgecasamayor/health-longevity-analytics/main/data/Mental_Health_Lifestyle_Dataset.csv"

# Download file locally (works outside Databricks CE)
local_path = "/tmp/Mental_Health_Lifestyle_Dataset.csv"
urllib.request.urlretrieve(url, local_path)

# Load into Spark DataFrame
# df = spark.read.csv(local_path, header=True, schema=schema)
# df.createOrReplaceTempView("mental_health_lifestyle_dataset")



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7223708161312867>:26
     23 urllib.request.urlretrieve(url, local_path)
     25 # 3️⃣ Load CSV into Spark DataFrame
---> 26 df = spark.read.csv(local_path, header=True, schema=schema)
     28 # 4️⃣ Check schema and sample data
     29 df.printSchema()

File /databricks/spark/python/pyspark/instrumentation_utils.py:48, in _wrap_function.<locals>.wrapper(*args, **kwargs)
     46 start = time.perf_counter()
     47 try:
---> 48     res = func(*args, **kwargs)
     49     logger.log_success(
     50         module_name, class_name, function_name, time.perf_counter() - start, signature
     51     )
     52     return res

File /databricks/spark/python/pyspark/sql/readwriter.py:729, in DataFrameReader.csv(self, path, schema, sep, encoding, quote, escape, comment, header, inferSchema, ignoreLeadingWhiteSpace, ignoreTrailingWhit

## Step 1. Load dataset from the Databricks metastore

In this step, we load the `mental_health_lifestyle_dataset` table
directly from the default Databricks database.
Since the table schema is already defined, we can immediately
inspect its structure and confirm that the data is available for analysis.



In [0]:
# Load the table from the default database
df = spark.table("default.mental_health_lifestyle_dataset_1_csv")

# Print schema to confirm data types
df.printSchema()

# Count total records
record_count = df.count()
print(f"Total records: {record_count}")

# Show sample data
df.show(5, truncate=False)



root
 |-- Country: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Exercise Level: string (nullable = true)
 |-- Diet Type: string (nullable = true)
 |-- Sleep Hours: double (nullable = true)
 |-- Stress Level: string (nullable = true)
 |-- Mental Health Condition: string (nullable = true)
 |-- Work Hours per Week: integer (nullable = true)
 |-- Screen Time per Day (Hours): double (nullable = true)
 |-- Social Interaction Score: double (nullable = true)
 |-- Happiness Score: double (nullable = true)

Total records: 3000
+---------+---+------+--------------+----------+-----------+------------+-----------------------+-------------------+---------------------------+------------------------+---------------+
|Country  |Age|Gender|Exercise Level|Diet Type |Sleep Hours|Stress Level|Mental Health Condition|Work Hours per Week|Screen Time per Day (Hours)|Social Interaction Score|Happiness Score|
+---------+---+------+--------------+-------

## Step 2. Data quality check – Missing values

Before starting any analysis, it is essential to check the data quality.  
Here we look for missing (NULL) values in each column of the dataset.  
This helps identify where cleaning or imputation might be needed.


In [0]:
from pyspark.sql.functions import col, sum

# Calculate number of missing values per column
missing_values = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_values.show(truncate=False)


+-------+---+------+--------------+---------+-----------+------------+-----------------------+-------------------+---------------------------+------------------------+---------------+
|Country|Age|Gender|Exercise Level|Diet Type|Sleep Hours|Stress Level|Mental Health Condition|Work Hours per Week|Screen Time per Day (Hours)|Social Interaction Score|Happiness Score|
+-------+---+------+--------------+---------+-----------+------------+-----------------------+-------------------+---------------------------+------------------------+---------------+
|0      |0  |0     |0             |0        |0          |0           |0                      |0                  |0                          |0                       |0              |
+-------+---+------+--------------+---------+-----------+------------+-----------------------+-------------------+---------------------------+------------------------+---------------+



### Interpretation

- Columns with **0 missing values** are fully complete.  
- Columns with **non-zero values** may require attention before further analysis.  
- In this case we have 0 missing values for all the fields


## Step 3. Descriptive statistics

In this step, we explore basic descriptive statistics for the main numerical variables.  
We use Spark SQL to calculate averages and standard deviations for age, sleep, work hours, screen time,  
social interaction, and happiness.

This helps us understand the general lifestyle trends in the dataset.


In [0]:
spark.sql("""
SELECT
    ROUND(AVG(Age), 1) AS avg_age,
    ROUND(STDDEV(Age), 1) AS std_age,
    ROUND(AVG(`Sleep Hours`), 2) AS avg_sleep,
    ROUND(STDDEV(`Sleep Hours`), 2) AS std_sleep,
    ROUND(AVG(`Work Hours per Week`), 1) AS avg_work,
    ROUND(AVG(`Screen Time per Day (Hours)`), 2) AS avg_screen,
    ROUND(AVG(`Social Interaction Score`), 2) AS avg_social,
    ROUND(AVG(`Happiness Score`), 2) AS avg_happiness,
    ROUND(STDDEV(`Happiness Score`), 2) AS std_happiness
FROM default.mental_health_lifestyle_dataset
""").show()


+-------+-------+---------+---------+--------+----------+----------+-------------+-------------+
|avg_age|std_age|avg_sleep|std_sleep|avg_work|avg_screen|avg_social|avg_happiness|std_happiness|
+-------+-------+---------+---------+--------+----------+----------+-------------+-------------+
|   41.2|   13.4|     6.48|      1.5|    39.5|      5.09|      5.47|          5.4|         2.56|
+-------+-------+---------+---------+--------+----------+----------+-------------+-------------+



## Step 3.1. Descriptive breakdown by gender

Next, we check how these lifestyle indicators differ between genders.  
This helps us understand if there are any notable patterns in habits or well-being.


In [0]:
# Display average happiness by gender
display(
    spark.sql("""
    SELECT Gender, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY Gender
    ORDER BY avg_happiness DESC
    """)
)



Gender,avg_happiness
Male,5.47
Other,5.43
Female,5.29


Databricks visualization. Run in Databricks to view.

### Step 3.2. Descriptive breakdown by country

Here we look at the top countries by average happiness score.  
This gives a quick overview of regional lifestyle differences.


In [0]:
display(
    spark.sql("""
    SELECT Country, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY Country
    HAVING COUNT(*) > 20
    ORDER BY avg_happiness DESC
    LIMIT 10
    """)
)



Country,avg_happiness
Canada,5.56
Australia,5.49
India,5.38
Germany,5.37
USA,5.35
Brazil,5.34
Japan,5.28


Databricks visualization. Run in Databricks to view.

## Step 4. Lifestyle factors and well-being

In this step, we explore how lifestyle variables such as exercise level, diet type,
and sleep hours relate to happiness and mental health condition.

We use Spark SQL to compute averages and distributions, 
and display results as interactive charts suitable for the dashboard.


### Average Happiness by Exercise Level

This bar chart shows how happiness varies with exercise frequency.
Higher activity levels may correspond to higher well-being.


In [0]:
display(
    spark.sql("""
    SELECT `Exercise Level`, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Exercise Level`
    ORDER BY avg_happiness DESC
    """)
)


Exercise Level,avg_happiness
High,5.55
Moderate,5.36
Low,5.29


Databricks visualization. Run in Databricks to view.

### Average Happiness by Diet Type

This bar chart explores whether dietary choices are associated with higher happiness.
It can reveal patterns between different diet types and well-being.


In [0]:
display(
    spark.sql("""
    SELECT `Diet Type`, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Diet Type`
    ORDER BY avg_happiness DESC
    """)
)


Diet Type,avg_happiness
Vegetarian,5.66
Junk Food,5.44
Keto,5.34
Vegan,5.29
Balanced,5.25


Databricks visualization. Run in Databricks to view.

### Sleep Hours Buckets vs Average Happiness

To better understand how different amounts of sleep affect well-being,
we group participants into 1-hour sleep buckets.
This allows us to see clear trends between sleep duration and happiness.



In [0]:
display(
    spark.sql("""
    SELECT 
        FLOOR(`Sleep Hours`) AS sleep_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY FLOOR(`Sleep Hours`)
    ORDER BY sleep_bucket
    """)
)



sleep_bucket,avg_happiness,count_participants
1,6.17,3
2,5.78,30
3,5.31,116
4,5.25,338
5,5.26,595
6,5.44,792
7,5.45,626
8,5.59,347
9,5.16,125
10,5.65,24


Databricks visualization. Run in Databricks to view.

### Stress Level vs Average Happiness

This bar chart shows the inverse relationship between stress levels and happiness.
Higher stress may correspond to lower well-being.



In [0]:
display(
    spark.sql("""
    SELECT `Stress Level`, ROUND(AVG(`Happiness Score`),2) AS avg_happiness
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Stress Level`
    ORDER BY avg_happiness ASC
    """)
)



Stress Level,avg_happiness
Moderate,5.34
Low,5.41
High,5.44


Databricks visualization. Run in Databricks to view.

### Work Hours Buckets vs Average Happiness

We group weekly work hours into 1-hour buckets to understand their effect on happiness.
This approach simplifies the visualization and highlights trends more clearly.


In [0]:
display(
    spark.sql("""
    SELECT 
        FLOOR(`Work Hours per Week`) AS work_hours_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY FLOOR(`Work Hours per Week`)
    ORDER BY work_hours_bucket
    """)
)


work_hours_bucket,avg_happiness,count_participants
20,5.43,64
21,5.77,76
22,5.43,70
23,5.85,83
24,5.74,66
25,4.49,57
26,5.44,94
27,5.4,88
28,5.37,68
29,5.07,61


Databricks visualization. Run in Databricks to view.

### Screen Time Buckets vs Average Happiness

We group daily screen time into 1-hour buckets to explore its relationship with happiness.
Bucketing helps to compare participants with different screen habits in a clear way.


In [0]:
display(
    spark.sql("""
    SELECT 
        FLOOR(`Screen Time per Day (Hours)`) AS screen_hours_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY FLOOR(`Screen Time per Day (Hours)`)
    ORDER BY screen_hours_bucket
    """)
)


screen_hours_bucket,avg_happiness,count_participants
2,5.47,449
3,5.27,481
4,5.34,470
5,5.33,501
6,5.4,531
7,5.54,537
8,5.42,31


Databricks visualization. Run in Databricks to view.

## Step 5. Combined Lifestyle Factors vs Happiness

In this step, we explore how multiple lifestyle factors interact to influence happiness.  
We group participants by:
- Exercise Level
- Diet Type
- Sleep Hours (buckets of 1 hour)

We calculate the average happiness for each combination to identify patterns and trends.


In [0]:
display(
    spark.sql("""
    SELECT
        `Exercise Level`,
        --`Diet Type`,
        FLOOR(`Sleep Hours`) AS sleep_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Exercise Level`, FLOOR(`Sleep Hours`)
    HAVING COUNT(*) >= 5
    ORDER BY `Exercise Level`, sleep_bucket
    """)
)


Exercise Level,sleep_bucket,avg_happiness,count_participants
High,2,6.4,5
High,3,4.72,38
High,4,5.11,102
High,5,5.57,182
High,6,5.67,269
High,7,5.54,221
High,8,5.83,104
High,9,6.0,40
High,10,3.33,6
Low,2,5.29,12


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    spark.sql("""
    SELECT
        --`Exercise Level`,
        `Diet Type`,
        FLOOR(`Sleep Hours`) AS sleep_bucket,
        ROUND(AVG(`Happiness Score`),2) AS avg_happiness,
        COUNT(*) AS count_participants
    FROM default.mental_health_lifestyle_dataset
    GROUP BY `Diet Type`, FLOOR(`Sleep Hours`)
    HAVING COUNT(*) >= 5
    ORDER BY `Diet Type`, sleep_bucket
    """)
)

Diet Type,sleep_bucket,avg_happiness,count_participants
Balanced,2,5.07,9
Balanced,3,5.33,27
Balanced,4,5.16,69
Balanced,5,4.92,136
Balanced,6,5.43,158
Balanced,7,5.64,136
Balanced,8,4.87,65
Balanced,9,4.53,20
Junk Food,2,6.59,10
Junk Food,3,6.31,21


Databricks visualization. Run in Databricks to view.

## Step 6. Heatmap: Sleep Buckets vs Happiness by Exercise Level

We create a pivot table to visualize average happiness across sleep buckets and exercise levels.  
This heatmap allows us to quickly spot trends and interactions between sleep and activity on well-being.


In [0]:
# Pivot table: rows = Sleep Bucket, columns = Exercise Level, values = avg_happiness
pivot_df = spark.sql("""
SELECT 
    FLOOR(`Sleep Hours`) AS sleep_bucket,
    `Exercise Level`,
    ROUND(AVG(`Happiness Score`),2) AS avg_happiness
FROM default.mental_health_lifestyle_dataset
GROUP BY FLOOR(`Sleep Hours`), `Exercise Level`
""").groupBy("sleep_bucket").pivot("Exercise Level").avg("avg_happiness")

# Display as interactive heatmap/table
display(pivot_df.orderBy("sleep_bucket"))


sleep_bucket,High,Low,Moderate
1,null,1.5,8.5
2,6.4,5.29,5.98
3,4.72,5.38,5.87
4,5.11,5.63,5.03
5,5.57,5.18,5.07
6,5.67,5.15,5.51
7,5.54,5.38,5.42
8,5.83,5.46,5.52
9,6.0,4.5,5.05
10,3.33,6.38,6.46


## Multivariate Analysis

In this section, we explore relationships between multiple variables to uncover patterns and insights.  
We will focus on combinations of lifestyle factors and mental health indicators.


In [0]:
# Import necessary functions
from pyspark.sql.functions import col, avg, count

# Example 1: Average Happiness Score by Exercise Level and Diet Type
exercise_diet_happiness = df.groupBy("Exercise Level", "Diet Type") \
                            .agg(avg("Happiness Score").alias("Avg_Happiness"),
                                 count("*").alias("Count")) \
                            .orderBy("Exercise Level", "Diet Type")

exercise_diet_happiness.show()

# Register as temporary view for SQL queries
df.createOrReplaceTempView("mental_health")


# **Dashboard visualizations**

# Univariate Analysis

In this section, we explore the distribution of the main variables in the dataset.  
These visualizations help understand the general patterns and trends in the data.


In [0]:
# Markdown
displayHTML("""
<b>Age Distribution</b><br>
Histogram showing the distribution of user ages.
""")

# PySpark: group by age and count
age_dist = df.groupBy("Age").count().orderBy("Age")

# Display as Databricks chart
display(age_dist)



Age Distribution 
Histogram showing the distribution of user ages.

Age,count
18,59
19,54
20,66
21,72
22,57
23,56
24,44
25,70
26,61
27,61


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Gender Proportion</b><br>
Pie chart showing the number of users by gender.
""")

gender_dist = df.groupBy("Gender").count()
display(gender_dist)


Gender Proportion 
Bar chart showing the number of users by gender.

Gender,count
Female,1024
Other,996
Male,980


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Exercise Level</b><br>
Bar chart showing the distribution of exercise levels among users.
""")

exercise_dist = df.groupBy("Exercise Level").count()
display(exercise_dist)


Exercise Level 
Bar chart showing the distribution of exercise levels among users.

Exercise Level,count
High,969
Low,1033
Moderate,998


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Diet Type</b><br>
Bar chart showing the distribution of diet types.
""")

diet_dist = df.groupBy("Diet Type").count()
display(diet_dist)


Diet Type 
Bar chart showing the distribution of diet types.

Diet Type,count
Keto,573
Balanced,625
Junk Food,637
Vegetarian,592
Vegan,573


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Sleep Hours</b><br>
Histogram showing the distribution of sleep hours.  
We round sleep hours to nearest integer for better visualization.
""")

sleep_dist = df.withColumn("Sleep_Hours_Rounded", df["Sleep Hours"].cast("int")) \
               .groupBy("Sleep_Hours_Rounded").count() \
               .orderBy("Sleep_Hours_Rounded")
display(sleep_dist)


Sleep Hours 
Histogram showing the distribution of sleep hours. 
We round sleep hours to nearest integer for better visualization.

Sleep_Hours_Rounded,count
1,3
2,30
3,116
4,338
5,595
6,792
7,626
8,347
9,125
10,24


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Happiness Score</b><br>
Histogram showing distribution of happiness scores.
""")

happiness_dist = df.withColumn("Happiness_Rounded", df["Happiness Score"].cast("int")) \
                   .groupBy("Happiness_Rounded").count() \
                   .orderBy("Happiness_Rounded")
display(happiness_dist)


Happiness Score 
Histogram showing distribution of happiness scores.

Happiness_Rounded,count
1,325
2,352
3,334
4,358
5,313
6,371
7,314
8,333
9,286
10,14


Databricks visualization. Run in Databricks to view.

# Bivariate Analysis

In this section, we explore relationships between two variables.  
These visualizations help understand how factors like exercise, diet, age, and gender relate to happiness and stress levels.


In [0]:
# Markdown
displayHTML("""
<b>Exercise Level vs Happiness Score</b><br>
Boxplot showing the distribution of Happiness Score for each Exercise Level.
""")

# Aggregate: compute average and count
exercise_happiness = df.groupBy("Exercise Level") \
                       .agg({"Happiness Score": "avg", "*": "count"}) \
                       .withColumnRenamed("avg(Happiness Score)", "Avg_Happiness") \
                       .withColumnRenamed("count(1)", "Count")

display(exercise_happiness)


Exercise Level vs Happiness Score 
Boxplot showing the distribution of Happiness Score for each Exercise Level.

Exercise Level,Count,Avg_Happiness
High,969,5.545407636738901
Low,1033,5.289545014520818
Moderate,998,5.358316633266541


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Diet Type vs Stress Level</b><br>
Bar chart showing counts of users per Stress Level for each Diet Type.
""")

diet_stress = df.groupBy("Diet Type", "Stress Level").count().orderBy("Diet Type")
display(diet_stress)


Diet Type vs Stress Level 
Bar chart showing counts of users per Stress Level for each Diet Type.

Diet Type,Stress Level,count
Balanced,Moderate,228
Balanced,High,204
Balanced,Low,193
Junk Food,Low,219
Junk Food,High,220
Junk Food,Moderate,198
Keto,Low,195
Keto,High,182
Keto,Moderate,196
Vegan,Low,206


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Age vs Happiness Score</b><br>
Scatter plot showing relationship between Age and Happiness Score.
""")

age_happiness = df.select("Age", "Happiness Score")
display(age_happiness)


Age vs Happiness Score 
Scatter plot showing relationship between Age and Happiness Score.

Age,Happiness Score
48,6.5
31,6.8
37,9.7
35,6.6
46,4.4
23,7.2
49,6.9
46,1.1
60,5.2
19,7.7


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Gender vs Happiness Score</b><br>
Boxplot showing the distribution of Happiness Score for each gender.
""")

# Directly select the needed columns, no aggregation
display(df.select("Gender", "Happiness Score"))


Gender vs Happiness Score 
Boxplot showing the distribution of Happiness Score for each gender.

Gender,Happiness Score
Male,6.5
Male,6.8
Female,9.7
Male,6.6
Male,4.4
Other,7.2
Male,6.9
Other,1.1
Male,5.2
Female,7.7


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Gender vs Stress Level</b><br>
Bar chart showing counts of each stress level by gender.
""")

stress_gender = df.groupBy("Gender", "Stress Level").count().orderBy("Gender")
display(stress_gender)


Gender vs Stress Level 
Bar chart showing counts of each stress level by gender.

Gender,Stress Level,count
Female,Low,355
Female,High,352
Female,Moderate,317
Male,Moderate,330
Male,Low,316
Male,High,334
Other,Moderate,343
Other,High,316
Other,Low,337


Databricks visualization. Run in Databricks to view.

# Multivariate Analysis

In this section, we explore relationships between multiple variables.  
We will show correlations among numeric variables and heatmaps for combinations of categorical variables to detect patterns.


In [0]:
# Markdown
displayHTML("""
<b>Correlation Matrix</b><br>
This heatmap shows the correlation between numeric variables:
Sleep Hours, Work Hours per Week, Screen Time per Day, Social Interaction Score, Happiness Score.
""")

# List of numeric columns
numeric_cols = ["Sleep Hours", "Work Hours per Week", "Screen Time per Day (Hours)",
                "Social Interaction Score", "Happiness Score"]

# Compute correlations
corr_data = [(c1, c2, df.stat.corr(c1, c2)) 
             for c1 in numeric_cols 
             for c2 in numeric_cols]

corr_df = spark.createDataFrame(corr_data, ["Var1", "Var2", "Correlation"])
display(corr_df)


Correlation Matrix 
This heatmap shows the correlation between numeric variables:
Sleep Hours, Work Hours per Week, Screen Time per Day, Social Interaction Score, Happiness Score.

Var1,Var2,Correlation
Sleep Hours,Sleep Hours,1.0
Sleep Hours,Work Hours per Week,0.011071087601014103
Sleep Hours,Screen Time per Day (Hours),0.0225500789905399
Sleep Hours,Social Interaction Score,-0.005221704509005462
Sleep Hours,Happiness Score,0.017388554289541884
Work Hours per Week,Sleep Hours,0.011071087601014077
Work Hours per Week,Work Hours per Week,1.0
Work Hours per Week,Screen Time per Day (Hours),-0.020282910906287806
Work Hours per Week,Social Interaction Score,0.015009114393564248
Work Hours per Week,Happiness Score,0.010837323614180118


Databricks visualization. Run in Databricks to view.

In [0]:
# Markdown
displayHTML("""
<b>Exercise Level + Diet Type vs Average Happiness Score</b><br>
Heatmap showing the average Happiness Score for each combination of Exercise Level and Diet Type.
""")

exercise_diet_happiness = df.groupBy("Exercise Level", "Diet Type") \
                            .agg({"Happiness Score": "avg"}) \
                            .withColumnRenamed("avg(Happiness Score)", "Avg_Happiness")

display(exercise_diet_happiness)


Exercise Level + Diet Type vs Average Happiness Score 
Heatmap showing the average Happiness Score for each combination of Exercise Level and Diet Type.

Exercise Level,Diet Type,Avg_Happiness
Low,Balanced,5.156444444444444
Low,Vegan,5.183980582524269
Moderate,Keto,5.10295566502463
High,Balanced,5.508499999999995
Low,Junk Food,5.174129353233831
Moderate,Vegetarian,5.509844559585493
High,Junk Food,5.544999999999999
Low,Keto,5.452083333333334
Moderate,Vegan,5.521084337349395
Moderate,Balanced,5.0894999999999975


Databricks visualization. Run in Databricks to view.